In [ ]:
# Check if a GPU is available and set the device
import tensorflow as tf

gpu_available = tf.config.list_physical_devices('GPU')
if gpu_available:
    print("GPU is available.")
else:
    print("No GPU available. Please ensure you have a GPU enabled in Colab settings.")

GPU is available.


In [ ]:
%pip install sentence-transformers numpy pandas

In [ ]:
from typing import Iterable, Iterator


# be careful not to make the context more important than the text
MAX_SUBJECT_LEN = 20
MAX_COMMITIEE_LEN = 20
# Corrected format string to properly embed subject and committee
METADATA_FORMAT = "[sub: %s comm: %s] %s"


def embed_metadata_in_utterance(
    utter_list: list[str], file_data: dict
):
    for u in utter_list:
        subject = file_data.get("subject", "Unknown Subject")[:MAX_SUBJECT_LEN]
        committee = file_data.get("committee", "Unknown Committee")[:MAX_COMMITIEE_LEN]
        yield METADATA_FORMAT % (subject, committee, u)


def strip_metadata_one(s: str) -> str:
    if s and s[0] == '[':
        i = s.find(']')
        if i != -1:
            return s[i+1:].lstrip()
    return s


def strip_metadata_many(utter_list: Iterable[str]) -> Iterator[str]:
    for u in utter_list:
        yield strip_metadata_one(u)

In [ ]:
import zipfile
import json
import os
import pandas as pd
from typing import Iterable, Iterator


# be careful not to make the context more important than the text
MAX_SUBJECT_LEN = 100
MAX_COMMITIEE_LEN = 100
# Corrected format string to properly embed subject and committee
METADATA_FORMAT = "[sub: %s comm: %s] %s"


def embed_metadata_in_utterance(
    utter_list: list[str], file_data: dict
):
    for u in utter_list:
        subject = file_data.get("subject", "Unknown Subject")[:MAX_SUBJECT_LEN]
        committee = file_data.get("committee", "Unknown Committee")[:MAX_COMMITIEE_LEN]
        yield METADATA_FORMAT % (subject, committee, u)


def strip_metadata_one(s: str) -> str:
    if s and s[0] == '[':
        i = s.find(']')
        if i != -1:
            return s[i+1:].lstrip()
    return s


def strip_metadata_many(utter_list: Iterable[str]) -> Iterator[str]:
    for u in utter_list:
        yield strip_metadata_one(u)


# Define the path to the zip file
zip_file_path = '/content/drive/MyDrive/utterances.zip' # Replace with the actual zip file path if different
extracted_dir = '/content/extracted_data'

# Create directory for extraction
os.makedirs(extracted_dir, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_dir)

all_utterances = []
utterances_df_list = []

# Function to process a single JSON file and extract utterances with metadata
def _process_utterance_file(file_name: str, filepath: str, utterances: list, utterances_df_list: list):
    with open(filepath, "r", encoding="utf-8") as file_content:
        file_data = json.loads(file_content.read())

        for speaker_key, values in file_data.get("utterances", {}).items():
            # Embed metadata directly when processing
            committee_prefixed_utterances = list(embed_metadata_in_utterance(
                values.get("utterances", []), file_data))

            utterances.extend(committee_prefixed_utterances)

            # Append data to the list for DataFrame creation
            for i, u in enumerate(values.get("utterances", [])):
                utterances_df_list.append(
                    {'text': u, "mk": speaker_key, "src": file_name, "utter_id": f"{file_name}_{speaker_key}_{i}"})


# Walk through the extracted directory and process each JSON file
print(f"Processing JSON files in {extracted_dir}...")
for root, _, files in os.walk(extracted_dir):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            _process_utterance_file(file, file_path, all_utterances, utterances_df_list)

print(f"Collected {len(all_utterances)} utterances with metadata.")
print(f"example: ")
print(f"{all_utterances[1]}")

# Create the DataFrame
utterances_df = pd.DataFrame(utterances_df_list)
print("\nDataFrame created:")
display(utterances_df.head())

# Save the DataFrame to a pickle file in Google Drive
utterances_df.to_pickle("/content/drive/MyDrive/utterances_data.pkl")
print("\nDataFrame saved to /content/drive/MyDrive/utterances_data.pkl")

Processing JSON files in /content/extracted_data...
Collected 1402179 utterances with metadata.
example: 
[sub: פרק ד' (מיסוי בשוק ה comm: ועדת_הכספים] מה ראשי התיבות של ריט?

DataFrame created:


,text,mk,src,utter_id
0,אני מחדש את הישיבה. אנחנו עוברים לפרק ד' בחוק ...,משה גפני,utterances_25_ptv_2430713.json,utterances_25_ptv_2430713.json_משה גפני_0
1,מה ראשי התיבות של ריט?,משה גפני,utterances_25_ptv_2430713.json,utterances_25_ptv_2430713.json_משה גפני_1
2,מה משמעות המילה ריט?,משה גפני,utterances_25_ptv_2430713.json,utterances_25_ptv_2430713.json_משה גפני_2
3,כל אחד יכול להשקיע?,משה גפני,utterances_25_ptv_2430713.json,utterances_25_ptv_2430713.json_משה גפני_3
4,מה אתם מציעים?,משה גפני,utterances_25_ptv_2430713.json,utterances_25_ptv_2430713.json_משה גפני_4



DataFrame saved to /content/drive/MyDrive/utterances_data.pkl


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import gc

# Load the SentenceTransformer model
model = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
)
mk_utternces = {}
# Define the embedding function
def _embed_in_vector_space(utternces: list, batch_size: int = 1000) -> np.ndarray:
    """
    Embed utterances in batches to reduce memory usage.
    """
    print(f"Encoding {len(utternces)} utterances in batches...")

    all_embeddings = []
    total_batches = (len(utternces) + batch_size - 1) // batch_size

    for i in range(0, len(utternces), batch_size):
        batch_end = min(i + batch_size, len(utternces))
        batch_utterances = utternces[i:batch_end]
        batch_embeddings = model.encode(batch_utterances, normalize_embeddings=True,convert_to_numpy=True, show_progress_bar=True)
        all_embeddings.append(batch_embeddings)
        print(f"Processed batch {i//batch_size + 1}/{total_batches}")

    embeddings_array = np.vstack(all_embeddings)

    # Clean up memory
    del all_embeddings
    gc.collect()

    print("Encoding completed!")
    return embeddings_array

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Embed the utterances
utterances_with_metadata = all_utterances
utterance_embeddings = _embed_in_vector_space(utterances_with_metadata)

# Save the embeddings to a .npy file
np.save("utterance_embeddings.npy", utterance_embeddings)

print("Embeddings saved to utterance_embeddings.npy")

Encoding 1402179 utterances in batches...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Processed batch 1/1403


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Processed batch 2/1403


...

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Processed batch 223/1403


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Processed batch 224/1403


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
import os
import shutil

source_path = "utterance_embeddings.npy"
destination_path = "/content/drive/MyDrive/utterance_embeddings.npy"

shutil.move(source_path, destination_path)
print(f"Moved {source_path} to {destination_path}")
